# 05. Jakość testów

## Cele

Po tym notebooku potrafisz:

* zamienić zgłoszenie błędu w test regresyjny i odróżnić szybką łatkę od usunięcia przyczyny,
* bezpiecznie refaktoryzować kod pod osłoną testów,
* zmierzyć pokrycie kodu (linii i gałęzi) i wiedzieć, czego ta liczba **nie** mówi,
* wyjaśnić ideę testowania mutacyjnego i TDD.

## Testy regresyjne

**Regresja** to błąd, który pojawia się w działającym wcześniej kodzie po zmianie w innym miejscu
([Wikipedia](https://pl.wikipedia.org/wiki/Regresja_(informatyka))). Najlepszym źródłem testów
regresyjnych są zgłoszenia błędów. Postępujemy według schematu:

1. **Odtwórz błąd testem.** Napisz test opisujący *poprawne* zachowanie z raportu i sprawdź, że **nie
   przechodzi** (czerwony). To dowód, że test rzeczywiście trafia w zgłoszony błąd — test, który
   przechodzi już przed poprawką, niczego nie sprawdza.
2. **Popraw kod** tak, żeby test przeszedł (zielony), a pozostałe testy nadal przechodziły.
3. **Zostaw test w zestawie na zawsze.** Jeśli ktoś kiedyś wprowadzi ten sam błąd ponownie, test go złapie.

W *Software Engineering at Google* tę zasadę streszcza „reguła Beyoncé”: *if you liked it, then you
shoulda put a test on it* — wszystko, czego nie chcesz zepsuć, powinno mieć test
([SWE at Google, rozdz. 11](https://abseil.io/resources/swe-book/html/ch11.html#the_beyonceacutesemicolon_rule)).
Nazwa testu może wskazywać na zgłoszenie, np. `TEST(BugReport42, ...)`, ale powinna też mówić,
jakie zachowanie sprawdza.

## Refaktoryzacja pod osłoną testów

**Refaktoryzacja** to zmiana wewnętrznej struktury kodu, która ułatwia jego zrozumienie i modyfikację,
**bez zmiany obserwowalnego zachowania**
([M. Fowler, *DefinitionOfRefactoring*](https://martinfowler.com/bliki/DefinitionOfRefactoring.html);
[Wikipedia](https://pl.wikipedia.org/wiki/Refaktoryzacja)).
Skąd wiadomo, że zachowanie się nie zmieniło? Z testów, które przechodziły przed zmianą
i przechodzą po niej ([*SelfTestingCode*](https://martinfowler.com/bliki/SelfTestingCode.html)).

Zasady:

* testy sprawdzają **publiczny interfejs** i zachowanie, a nie szczegóły implementacji — wtedy zmiana
  wnętrza klasy nie wymaga zmian w testach,
* zmieniaj kod **małymi krokami** i uruchamiaj testy po każdym kroku,
* jeśli test trzeba zmienić, to znaczy, że zmienia się zachowanie (albo test był związany z implementacją),
* dla starego kodu bez testów najpierw pisze się *testy charakteryzujące*, które utrwalają obecne
  zachowanie ([Characterization test](https://en.wikipedia.org/wiki/Characterization_test)).

Katalog nazwanych refaktoryzacji: [refactoring.com/catalog](https://refactoring.com/catalog/) — np.
*Replace Primitive with Object*.

## Pokrycie kodu

**Pokrycie kodu** (*code coverage*) mówi, jaka część kodu została **wykonana** podczas testów:

* **pokrycie linii** — ile linii wykonano co najmniej raz,
* **pokrycie gałęzi** — ile gałęzi decyzji (np. „prawda” i „fałsz” każdego `if`) wykonano.

Te liczby mogą się różnić. Jeden test `discountPercent(true, true)` wykona każdą linię poniższej funkcji
(100% linii), ale nigdy nie sprawdzi przypadku, w którym warunek jest fałszywy (50% gałęzi):

```cpp
int discountPercent(bool isStudent, bool hasCard) {
    int discount = 0;
    if (isStudent && hasCard) discount = 50;
    return discount;
}
```

### Jak zmierzyć pokrycie (GCC / Clang)

1. Skompiluj testy z flagą `--coverage` (dla GCC to skrót od `-fprofile-arcs -ftest-coverage` przy
   kompilacji i `-lgcov` przy linkowaniu, [GCC](https://gcc.gnu.org/onlinedocs/gcc/Instrumentation-Options.html#index-coverage)).
2. Uruchom testy — program zapisze pliki `.gcda` z licznikami wykonań.
3. Wygeneruj raport narzędziem [gcovr](https://gcovr.com/en/stable/guide/compiling.html)
   albo lcov. gcovr to pakiet Pythona, już zainstalowany w środowisku laboratorium (zależność
   w `pyproject.toml`). Pokrycie gałęzi: opcja `--txt-metric branch`.

Kod C++ generuje też gałęzie niewidoczne w źródle (wyjątki, destruktory), więc pokrycie gałęzi
dla całego programu bywa zaskakująco niskie ([gcovr FAQ](https://gcovr.com/en/stable/faq.html#exception-branches)).
W Visual Studio pokrycie mierzy się innymi narzędziami — ta demonstracja działa z GCC (Linux, WSL, MinGW).

Demonstracja: [`demos/ticket.cpp`](demos/ticket.cpp) liczy cenę biletu (dzieci poniżej 7 lat — za darmo,
seniorzy od 65 lat — połowa ceny), a [`demos/ticket_test.cpp`](demos/ticket_test.cpp) zawiera trzy testy.

In [ ]:
# Build the demo with coverage instrumentation and run the tests
!cmake -S demos -B demos/build -DENABLE_COVERAGE=ON && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug --output-on-failure

In [ ]:
# Coverage report: lines, then branches (gcovr runs with the kernel's Python)
import sys
!{sys.executable} -m gcovr --root demos --filter demos/ticket.cpp demos/build && {sys.executable} -m gcovr --root demos --filter demos/ticket.cpp --txt-metric branch demos/build

### Czego pokrycie nie mówi

Raport pokazuje 100% linii i 100% gałęzi. Czy to znaczy, że funkcja jest dobrze przetestowana?
Pokrycie mówi tylko, że kod **został wykonany**, a nie że jego wynik **został sprawdzony**. Test bez
żadnej asercji też daje pokrycie. Google pisze wprost, że pokrycie nie gwarantuje, że pokryte linie
i gałęzie zostały przetestowane poprawnie
([Google Testing Blog — *Code Coverage Best Practices*](https://testing.googleblog.com/2020/08/code-coverage-best-practices.html)),
a M. Fowler — że jako liczbowa miara jakości testów pokrycie jest mało przydatne; jest za to dobrym
sposobem na znalezienie **nieprzetestowanych** miejsc ([*TestCoverage*](https://martinfowler.com/bliki/TestCoverage.html)).
Gdy wysokie pokrycie staje się celem samym w sobie, łatwo o testy, które wykonują kod, ale niczego
nie sprawdzają ([SWE at Google, rozdz. 11](https://abseil.io/resources/swe-book/html/ch11.html#a_note_on_code_coverage)).
Dla orientacji: według tego samego wpisu Google 60% to poziom „akceptowalny”, 75% — „godny pochwały”,
90% — „wzorowy”.

## Testowanie mutacyjne

Jak sprawdzić, czy testy naprawdę *sprawdzają*? **Testowanie mutacyjne** wprowadza do kodu drobne,
celowe błędy — **mutanty** (np. `<` → `<=`, `+` → `-`, usunięcie linii) — i uruchamia testy dla każdego
z nich. Jeśli jakiś test nie przechodzi, mutant jest **zabity**. Jeśli wszystkie testy przechodzą,
mutant **przeżył**: znaleźliśmy zmianę zachowania, której testy nie zauważają. Odsetek zabitych mutantów
to **wynik mutacyjny** (*mutation score*)
([Google Testing Blog — *Mutation Testing*](https://testing.googleblog.com/2021/04/mutation-testing.html);
[Wikipedia](https://pl.wikipedia.org/wiki/Testowanie_mutacyjne)).

**Spróbuj ręcznie:** w pliku [`demos/ticket.cpp`](demos/ticket.cpp) zmień `age < 7` na `age <= 7`,
zapisz plik i uruchom ponownie pierwszą komórkę demonstracji. Testy nadal przechodzą, a pokrycie nadal
wynosi 100% — mutant przeżył. Których testów brakuje? (Wskazówka: notebook 1, wartości brzegowe.)
Dopisz je w [`demos/ticket_test.cpp`](demos/ticket_test.cpp) tak, by zabić ten mutant i analogiczny
mutant `age >= 65` → `age > 65`. Na koniec przywróć `age < 7`.

Narzędzia robią to automatycznie dla setek mutantów. Dla C++ najbardziej aktywnie rozwijanym jest
[Mull](https://mull.readthedocs.io/en/latest/) (oparty na Clang/LLVM; krótki pokaz:
[C++ Weekly, odc. 455](https://www.youtube.com/watch?v=lhcXAnNgzlo)). Testowanie mutacyjne jest
kosztowne obliczeniowo, dlatego np. Google stosuje je tylko do zmienianych fragmentów kodu
([*State of Mutation Testing at Google*](https://research.google/pubs/state-of-mutation-testing-at-google/)).

## TDD — najpierw test

**Test-Driven Development** (K. Beck, koniec lat 90.) odwraca kolejność: test powstaje *przed* kodem.
Cykl jest często streszczany jako **Red – Green – Refactor**
([M. Fowler, *TestDrivenDevelopment*](https://martinfowler.com/bliki/TestDrivenDevelopment.html)):

1. **Red** — napisz mały test nowego zachowania i zobacz, że nie przechodzi,
2. **Green** — napisz najprostszy kod, który sprawia, że test przechodzi,
3. **Refactor** — popraw strukturę kodu (i testów), utrzymując wszystkie testy na zielono.

K. Beck dodaje krok zerowy: zacznij od **listy przypadków testowych** i przerabiaj je po jednym
([*Canon TDD*](https://newsletter.kentbeck.com/p/canon-tdd)). R. C. Martin formułuje to ostrzej jako
„trzy prawa TDD” ([*The Three Rules of TDD*](http://butunclebob.com/ArticleS.UncleBob.TheThreeRulesOfTdd)).
TDD gwarantuje, że każdy fragment kodu ma test, który kiedyś nie przechodził — czyli test, który
naprawdę coś sprawdza. Schemat testu regresyjnego z początku notebooka to w gruncie rzeczy jeden cykl TDD.

Do przećwiczenia w domu: kata *Bowling Game* — opis w notebooku 6.

## Ćwiczenie

*Treść ćwiczenia — w przygotowaniu.*

## Linki i materiały

* [M. Fowler — *Refactoring*](https://martinfowler.com/books/refactoring.html) i [katalog refaktoryzacji](https://refactoring.com/catalog/)
* [Google Testing Blog — *Code Coverage Best Practices*](https://testing.googleblog.com/2020/08/code-coverage-best-practices.html)
* [gcovr — dokumentacja](https://gcovr.com/en/stable/guide/compiling.html)
* [Google Testing Blog — *Mutation Testing*](https://testing.googleblog.com/2021/04/mutation-testing.html)
* [P. Nash — *Test Driven C++*, CppCon 2020 (wideo)](https://www.youtube.com/watch?v=N2gTxeIHMP0)